# Capstone Track B — Fine-Tuned Model Showcase
**Day 2 Afternoon | ~3 hours | Colab T4 GPU**

> **Switch to a T4 first:** Runtime → Change runtime type → T4 GPU → Save. Then run from the top.

---

In Lab 4 you trained an adapter on a dataset we wrote. Here you choose the task, write the examples, pick the rank, and then show a room full of people that the model behaves differently afterwards. That last part is the hard one. A falling loss proves the maths ran; it does not prove the model learned anything you care about.

**Your decisions, marked `TODO` in the cells:**

- [ ] Step 1: A narrow task, and 15 to 25 examples of it
- [ ] Step 3: The LoRA rank, and why
- [ ] Step 5: Held-out prompts that make the difference obvious
- [ ] Step 6: Three example prompts for the app

Loading, quantization, training and the Gradio shell are provided, using the same patterns as Lab 4.

> **Track A (RAG)?** Open `capstone_track_a.ipynb` instead. It runs on a free CPU runtime.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} "transformers>=5" torch accelerate bitsandbytes peft "trl>=0.16" datasets "gradio>=6"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU → Save, then run from the top."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_ID    = "Qwen/Qwen2.5-0.5B-Instruct"   # fits a T4 with room to spare; 1.5B also fits (Lab 4)
OUTPUT_DIR  = "./capstone_lora"
ADAPTER_DIR = "./capstone_adapter"

---

## Step 1 — Choose a task and write examples

**Narrow tasks show a clear before and after.** The base model already answers general questions well, so "be more helpful" gives you nothing to show. You want a behaviour the base model does *not* have and your examples teach consistently.

Tasks that work:
- Always answer in one specific JSON schema
- Rewrite technical error messages in plain English for non-developers
- Classify text into categories you invented
- Write emails or letters with a fixed structure
- Answer in a house style: a set opening, a length limit, a disclaimer at the end

Tasks that don't: anything vague, and anything the base model already does.

**Write 15 to 25 examples**, every one in the same pattern. Quality beats quantity: ten consistent examples teach more than thirty inconsistent ones. The last three are held out. The model never trains on them, and Step 5 tests on them.

In [ ]:
# TODO: describe your task and replace every example
TASK_DESCRIPTION = "DESCRIBE YOUR TASK HERE"
# e.g. "Given a technical error message, explain it in plain English for a non-developer."

training_examples = [
    {"instruction": "EXAMPLE INPUT 1", "response": "IDEAL OUTPUT 1"},
    {"instruction": "EXAMPLE INPUT 2", "response": "IDEAL OUTPUT 2"},
    {"instruction": "EXAMPLE INPUT 3", "response": "IDEAL OUTPUT 3"},
    {"instruction": "EXAMPLE INPUT 4", "response": "IDEAL OUTPUT 4"},
    {"instruction": "EXAMPLE INPUT 5", "response": "IDEAL OUTPUT 5"},
    # ... keep going to 15-25
]

train_data = training_examples[:-3]     # the model learns from these
test_data  = training_examples[-3:]     # and is tested on these, never trained on them

print(f"Task: {TASK_DESCRIPTION}")
print(f"Train: {len(train_data)}   Held out: {len(test_data)}")

---

## Step 2 — Load the base model in 4 bits

The same `BitsAndBytesConfig` as Lab 4, with one change: the compute dtype is `float16`, because a T4 does fp16 natively and only emulates bf16.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
print(f"Loaded {MODEL_ID}. VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

One helper for every generation in this notebook, the same shape as Lab 4's `ask()`. It formats with the chat template as a string, tokenizes that string, and decodes only the new tokens. `do_sample=False` means greedy decoding, so a difference between two answers comes from the weights, not from the dice.

In [ ]:
def ask(m, prompt, max_new_tokens=200):
    formatted = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                              tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Record the base model's answers on the held-out prompts BEFORE any training
base_answers = {ex["instruction"]: ask(model, ex["instruction"]) for ex in test_data}

first = test_data[0]["instruction"]
print("Prompt:", first)
print("Base  :", base_answers[first])

**Checkpoint:** this is your "before". Keep it. Once Step 3 attaches adapters, `model` is no longer the plain base model: `get_peft_model` adds the LoRA layers to it in place.

---

## Step 3 — Attach LoRA adapters

The rank `r` sets how many parameters you train. These are **measured** on this model (Qwen2.5-0.5B, 494M parameters):

| Rank | `q_proj` + `v_proj` only | every linear layer (Lab 4's choice) |
|------|------------------------------|--------------------------------------|
| 4    | 0.27M params, 0.06%, ~1 MB | 2.2M, 0.44%, ~9 MB |
| 8    | 0.54M params, 0.11%, ~2 MB | 4.4M, 0.88%, ~18 MB |
| 16   | 1.08M params, 0.22%, ~4 MB | 8.8M, 1.75%, ~35 MB |

(Sizes are fp32 weights. The saved adapter adds a little for config and metadata.)

More trainable parameters let the adapter learn more, and with 15 to 25 examples they also let it memorize more. A narrow format task usually does fine on the left column. A task that changes tone or vocabulary across the whole answer may need the right.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False                 # training, not generating (Lab 4 explains why)
model = prepare_model_for_kbit_training(model)

# TODO: choose rank and target modules, and justify them
# My task is [X], so I chose rank [Y] on [Z] because [...]
LORA_RANK   = 8
LORA_TARGET = ["q_proj", "v_proj"]             # or "all-linear"

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK,      # alpha = 2 x rank, as in Lab 4
    target_modules=LORA_TARGET,
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

`print_trainable_parameters()` counts the 4-bit weights correctly. A plain `sum(p.numel())` would not: bitsandbytes packs two 4-bit weights into each stored byte, so the total would come out at roughly two thirds of the real model.

---

## Step 4 — Fine-tune

Each example becomes one string through the chat template, with `add_generation_prompt=False` because the assistant's answer is already in it. Then the same `SFTConfig` / `SFTTrainer` as Lab 4.

In [ ]:
from datasets import Dataset

def format_example(ex):
    messages = [{"role": "user", "content": ex["instruction"]},
                {"role": "assistant", "content": ex["response"]}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = Dataset.from_list(train_data).map(format_example, remove_columns=["instruction", "response"])
print(train_dataset[0]["text"][:300])

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,                       # T4-native; Lab 4's bf16 would be emulated here
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    max_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_dataset, processing_class=tokenizer)
trainer.train()
model.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

**Checkpoint:** the loss went down. On 15 to 25 examples it will go down a long way, because memorizing a handful of strings is easy. That tells you the adapter is training. Whether it learned the *task* is what Step 5 is for, on prompts it never saw.

---

## Step 5 — Before and after

`model` now carries your adapter. First, `model.eval()`. The trainer leaves the model in training mode with gradient checkpointing on, and in that state `generate()` quietly throws away the KV cache and produces garbage. Then compare its answers on the held-out prompts with the base answers you saved in Step 2.

`model.disable_adapter()` is worth knowing too: inside that `with` block the same object behaves as the plain base model. One model in memory, adapter on or off. That is how servers host many adapters over one base.

In [ ]:
model.eval()                          # leave training mode: switches off gradient checkpointing and dropout
model.config.use_cache = True         # generating again, so the KV cache is back on

for ex in test_data:
    prompt = ex["instruction"]
    print("PROMPT  :", prompt)
    print("EXPECTED:", ex["response"][:150])
    print("BASE    :", base_answers[prompt][:150])
    print("TUNED   :", ask(model, prompt)[:150])
    print("-" * 70)

Read every row. You are looking for a difference someone at the back of the room can see in two seconds. If it is subtle:

- Make the examples more consistent with each other. This fixes more than anything else.
- Add examples, or narrow the task.
- Try `"all-linear"` or a higher rank.
- Go to 5 epochs, and watch for the model repeating training answers word for word.

Write down one prompt where the tuned model is *worse* than the base, or no better. That goes in your presentation.

---

## Step 6 — The comparison app

Two answers side by side for any prompt. The base column uses `disable_adapter()`, so both columns come from the one model already in memory.

In [ ]:
import gradio as gr

# TODO: three prompts that show your task best
EXAMPLE_PROMPTS = [
    "YOUR TEST PROMPT 1",
    "YOUR TEST PROMPT 2",
    "YOUR TEST PROMPT 3",
]

def compare(prompt):
    with model.disable_adapter():
        base_out = ask(model, prompt)
    return base_out, ask(model, prompt)

with gr.Blocks(title="Before and after") as demo:
    gr.Markdown(f"# Fine-tuning showcase\n**Task:** {TASK_DESCRIPTION}")
    prompt_box = gr.Textbox(label="Prompt", lines=3)
    with gr.Row():
        base_box  = gr.Textbox(label=f"Base: {MODEL_ID}", lines=10, interactive=False)
        tuned_box = gr.Textbox(label="Fine-tuned (QLoRA)", lines=10, interactive=False)
    gr.Button("Compare", variant="primary").click(compare, [prompt_box], [base_box, tuned_box])
    gr.Examples(EXAMPLE_PROMPTS, inputs=prompt_box)

demo.launch(share=True, quiet=True)

---

## Submission checklist

- [ ] The task is narrow, and the before/after difference is visible without explanation
- [ ] At least 15 examples, with 3 held out and never trained on
- [ ] You can say what rank and target modules you chose, and why
- [ ] The comparison app is live on a public link
- [ ] You have one prompt where fine-tuning did not help, and a guess at why

**Presentation (5 minutes):**

1. *"I fine-tuned [model] to [task] because [reason]."* (30 seconds)
2. The pipeline: examples → chat template → 4-bit base → LoRA adapters → `SFTTrainer` → saved adapter. Say what you chose at each step. (60 seconds)
3. Live demo: two prompts that show the difference, one that doesn't. (2 minutes)
4. What surprised you, and what you would change. (90 seconds)